# Notebook 01: The Attention Mechanism

## Why This Matters

Attention is the single most important idea in modern deep learning. Every large language model, image generation model, and multimodal system in production today is built on it. When a model like GPT-4 reads a paragraph and answers a question about it, attention is the mechanism that routes information from the right tokens to the right positions. Understanding attention from first principles — not just the API call — is the difference between being a practitioner who uses transformers and an engineer who can design, debug, and improve them. In interviews at ML-forward companies, "explain attention" is the most common architecture question, and the depth of your answer signals your level clearly.

## Background

Before attention, sequence models were dominated by RNNs (LSTMs, GRUs). The fundamental flaw: **a fixed-size hidden state had to compress the entire source sequence**. For a 200-word sentence, that single vector bottleneck caused the famous "long-range dependency problem" — early tokens were forgotten by the time the decoder needed them.

The Bahdanau (2015) attention paper introduced a partial fix: let the decoder look back at **all** encoder hidden states and form a weighted sum. This was a huge improvement, but it was bolted onto an RNN. The "Attention Is All You Need" paper (Vaswani et al. 2017) asked: what if we remove the RNN entirely and use attention as the primary computation? The result is the Transformer.

**The retrieval analogy:** Think of attention as a soft database lookup.
- **Query (Q):** what you're looking for (the current token's "question")
- **Key (K):** what each position advertises it contains (index entries)
- **Value (V):** the actual content to retrieve (the data)

The similarity between a query and each key determines how much of each value gets mixed in.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Scaled Dot-Product Attention

The core formula is elegant:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Why scale by $\sqrt{d_k}$?** If $d_k = 64$, the dot products $q \cdot k$ have magnitude ~$\sqrt{64} = 8$ (since each dimension contributes variance ~1 if weights are initialized correctly). Without scaling, large dot products push softmax into saturation regions where gradients vanish. Dividing by $\sqrt{d_k}$ keeps variance at ~1 regardless of head dimension.

**Common interview question:** "Why do we use softmax here?"  
**Answer:** Softmax converts raw similarity scores into a probability distribution over positions. This means each output is a convex combination (weighted average) of values, where weights sum to 1. It creates a competition: attending to one position reduces attention to all others — implementing a soft routing / selection mechanism.

**What the attention matrix $A = \text{softmax}(QK^T/\sqrt{d_k})$ represents:** Each row $i$ tells us, "when processing position $i$, how much information to pull from position $j$." The matrix has shape $(T \times T)$ where $T$ is sequence length.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q: (batch, heads, seq_q, d_k)
        K: (batch, heads, seq_k, d_k)
        V: (batch, heads, seq_k, d_v)
        mask: (batch, 1, seq_q, seq_k) or (1, 1, seq_q, seq_k), True = mask out
    Returns:
        output: (batch, heads, seq_q, d_v)
        attn_weights: (batch, heads, seq_q, seq_k)
    """
    d_k = Q.size(-1)
    # (batch, heads, seq_q, seq_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # Fill masked positions with -inf so softmax → 0
        scores = scores.masked_fill(mask, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)

    # Weighted sum of values
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Quick sanity check
B, T, d_k = 2, 6, 64
Q = torch.randn(B, 1, T, d_k)
K = torch.randn(B, 1, T, d_k)
V = torch.randn(B, 1, T, d_k)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {out.shape}")       # (2, 1, 6, 64)
print(f"Weights shape: {w.shape}")        # (2, 1, 6, 6)
print(f"Weights sum to 1: {w[0,0].sum(dim=-1)}")  # all ~1.0

## 2. Attention Masks: Causal and Padding

Two types of masks are essential in practice:

**Causal mask (autoregressive / GPT-style):** Position $i$ can only attend to positions $\leq i$. This is implemented as an upper-triangular matrix of $-\infty$. Without it, the model would "see the future" during training, learning a trivial copy task rather than next-token prediction.

**Padding mask:** Batches have variable-length sequences. Shorter sequences are padded to the max length. Padding positions should not contribute to attention — we mask them out.

**Interview question:** "What's the difference between the encoder mask and decoder mask?"  
**Answer:** The encoder uses a padding mask only (all positions can see all real positions). The decoder uses a causal mask (each position only sees past positions) PLUS a padding mask. During cross-attention in encoder-decoder models, the decoder query attends to all valid encoder key positions.

In [ ]:
def make_causal_mask(seq_len, device='cpu'):
    """Upper-triangular mask: position i cannot attend to j > i."""
    # True = mask out
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

def make_padding_mask(lengths, max_len, device='cpu'):
    """lengths: (B,) tensor of valid token counts. Returns (B,1,1,T) mask."""
    idx = torch.arange(max_len, device=device).unsqueeze(0)  # (1, T)
    mask = idx >= lengths.unsqueeze(1)  # (B, T): True where padded
    return mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)

# Demo causal mask
T = 6
causal = make_causal_mask(T)
print("Causal mask (True = blocked):")
print(causal[0, 0].int().numpy())

# Demo padding mask
lengths = torch.tensor([6, 4, 5])
pad_mask = make_padding_mask(lengths, max_len=6)
print("\nPadding mask for seq of length 4 (row 1):")
print(pad_mask[1, 0, 0].int().numpy())

## 3. Multi-Head Attention

A single attention head learns one type of relationship. **Multi-head attention** runs $h$ attention heads in parallel, each with its own learned $W^Q_i, W^K_i, W^V_i$ projections into a smaller dimension $d_k = d_{\text{model}} / h$.

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{where head}_i = \text{Attention}(QW^Q_i, KW^K_i, VW^V_i)$$

**Why multiple heads?** Different heads specialize in different relationship types. In practice, some heads learn syntactic dependencies (subject-verb agreement), some semantic similarity, some coreference. With a single head, the model must average all these patterns into one attention distribution — a loss of expressiveness.

**Parameter count:** For $d_{\text{model}} = 512$, $h = 8$ heads, $d_k = d_v = 64$:
- $W^Q$: $512 \times 512 = 262{,}144$ (covers all heads)
- $W^K$: $512 \times 512$
- $W^V$: $512 \times 512$  
- $W^O$: $512 \times 512$
- **Total per MHA layer: $4 \times 512^2 \approx 1M$ parameters**

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # Single matrix projections (more efficient than n_heads separate ones)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """(B, T, d_model) → (B, h, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.n_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        _, T_k, _ = key.shape

        Q = self.split_heads(self.W_q(query))   # (B, h, T_q, d_k)
        K = self.split_heads(self.W_k(key))     # (B, h, T_k, d_k)
        V = self.split_heads(self.W_v(value))   # (B, h, T_k, d_k)

        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        # (B, h, T_q, d_k) → (B, T_q, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        output = self.W_o(attn_out)
        return output, attn_weights

# Test
d_model, n_heads, T, B = 128, 8, 10, 4
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(B, T, d_model)
out, weights = mha(x, x, x)
print(f"MHA output shape: {out.shape}")         # (4, 10, 128)
print(f"Attention weights shape: {weights.shape}")  # (4, 8, 10, 10)

param_count = sum(p.numel() for p in mha.parameters())
print(f"Total MHA parameters: {param_count:,}")  # 4 * 128^2 = 65,536

## 4. Visualizing Attention Patterns

Attention patterns are the interpretability window into transformer behavior. By inspecting $A_{ij}$ — how much position $i$ attends to position $j$ — we can observe:

- **Diagonal patterns**: each token attends mostly to itself (common in middle layers)
- **Causal stripes**: each token attends to recent past (local context)  
- **Global tokens**: CLS tokens aggregating information from all positions
- **Syntactic heads**: attending to syntactic parent/child relationships

The famous "attention head analysis" papers (Clark et al. 2019, Voita et al. 2019) identified specific heads in BERT that reliably track syntactic roles — evidence that functional specialization emerges from gradient descent alone.

In [ ]:
# Visualize attention patterns with a causal mask
T = 12
d_model, n_heads = 64, 4
mha_vis = MultiHeadAttention(d_model, n_heads)
x_vis = torch.randn(1, T, d_model)
causal_mask = make_causal_mask(T)

with torch.no_grad():
    _, attn_weights = mha_vis(x_vis, x_vis, x_vis, mask=causal_mask)

fig, axes = plt.subplots(1, n_heads, figsize=(14, 3.5))
fig.suptitle("Multi-Head Attention Patterns (Causal Mask, Random Weights)", fontsize=13)
for h in range(n_heads):
    ax = axes[h]
    w = attn_weights[0, h].detach().numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    ax.set_title(f"Head {h+1}")
    ax.set_xlabel("Key position (attended to)")
    ax.set_ylabel("Query position")
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
plt.colorbar(im, ax=axes[-1], fraction=0.046)
plt.tight_layout()
plt.savefig("attention_patterns.png", dpi=120, bbox_inches='tight')
plt.show()
print("Note: upper triangle is -inf masked → 0 after softmax (causal attention)")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Scaled dot-product attention | $\text{softmax}(QK^T/\sqrt{d_k})V$; scaling prevents softmax saturation |
| Query / Key / Value | Q = what to look for; K = what positions advertise; V = content to retrieve |
| Why $\sqrt{d_k}$ scaling | Keeps dot-product variance ~1, preventing gradient vanishing in softmax |
| Causal mask | Upper-triangular -inf mask; prevents attending to future positions |
| Padding mask | Masks out padding tokens from attention scores |
| Multi-head attention | $h$ parallel heads in subspaces; each learns different relationship types |
| Head specialization | Different heads empirically track syntax, semantics, coreference |
| MHA parameter count | $4 \times d_{\text{model}}^2$ (Q, K, V, O projections) |
| Attention complexity | $O(T^2 d)$ time and $O(T^2)$ memory — the bottleneck for long sequences |